In [ ]:
# Input: 7762_Predicted_metabolites.csv with index and TA, IPA, INDOLE, ILA, IE, I3C columns.
# Input: 7762_Metadata.xlsx with matching index, Cohort, study_condition, age, gender, and BMI columns; one record per sample.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import MultipleLocator
from scipy.stats import norm, t as t_dist, chi2

BASE_DIR = Path()

DISEASE_ORDER = ["CRC", "IBD", "adenoma", "T2D"]
MIN_ELIGIBLE_COHORTS = 2

COHORT_DISPLAY = {
    "HMP_IBD": "HMP2",
    "HallAB_2017": "Hall, A. B.",
    "IBD_Pilot": "HMP2 Pilot",
    "FengQ_2015": "Feng et al.",
    "ThomsAMa_2018": "Thoms et al.",
    "VogtmannE_2016": "Vogtmann et al.",
    "YachidaS_2019": "Yachida et al.",
    "YuJ_2015": "Yu et al.",
    "KarlssonFH_2013": "Karlsson et al.",
    "QinJ_2012": "Qin et al.",
}

STUDY_MARKER_SET = ["o", "^", "s", "D", "v"]

FEATURE_ORDER = ["TA", "IPA", "INDOLE", "ILA", "IE", "IALD"]
FEATURE_SOURCE = {"IALD": "I3C"}

Q_CUT = 0.10
XLIM = (-1.25, 1.50)
CSV_PATH = BASE_DIR / "7762_Predicted_metabolites.csv"
METADATA_PATH = BASE_DIR / "7762_Metadata.xlsx"
OUTPUT_STEM = "multi_cohort_heterogeneity"

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["svg.fonttype"] = "none"


In [ ]:
CONTROL_LABELS = {
    "control", "ctrl", "healthy", "normal", "healthy control", "hc"
}
CASE_LABELS = {
    "case", "ibd", "cd", "crohn", "crohn's disease", "crohns disease",
    "uc", "ulcerative colitis"
}

def map_group(x):

    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in CONTROL_LABELS:
        return "Control"
    if s in CASE_LABELS:
        return "Case"
    return np.nan

def canonical_study(x):

    if pd.isna(x):
        return np.nan
    return COHORT_ALIASES.get(str(x).strip(), np.nan)

def normalize_condition(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()

def disease_subset(data, disease):

    condition = data["study_condition"].map(normalize_condition)
    target = str(disease).strip().lower()
    keep = condition.isin({"control", target})
    out = data.loc[keep].copy()
    out["_grp_"] = np.where(condition.loc[keep].eq("control"), "Control", "Case")
    out["_study_"] = out["Cohort"].astype(str).str.strip()
    return out

def gender_to_binary(series):

    mapping = {
        "female": 0.0, "f": 0.0, "0": 0.0,
        "male": 1.0, "m": 1.0, "1": 1.0,
    }
    s = series.astype(str).str.strip().str.lower()
    out = s.map(mapping)
                                                                             
    numeric = pd.to_numeric(series, errors="coerce")
    out = out.where(out.notna(), numeric.where(numeric.isin([0, 1])))
    return out.astype(float)

def hedges_g(vals_ctrl, vals_case):

    x0 = np.asarray(vals_ctrl, dtype=float)
    x1 = np.asarray(vals_case, dtype=float)
    x0 = x0[np.isfinite(x0)]
    x1 = x1[np.isfinite(x1)]
    n0, n1 = len(x0), len(x1)
    if n0 < 2 or n1 < 2:
        return np.nan, np.nan

    s0 = np.std(x0, ddof=1)
    s1 = np.std(x1, ddof=1)
    sp2 = ((n0 - 1) * s0**2 + (n1 - 1) * s1**2) / (n0 + n1 - 2)
    if not np.isfinite(sp2) or sp2 <= 0:
        return np.nan, np.nan

    d = (np.mean(x1) - np.mean(x0)) / np.sqrt(sp2)
    J = 1 - 3 / (4 * (n0 + n1) - 9)
    g = J * d
    v = (n0 + n1) / (n0 * n1) + (g**2) / (2 * (n0 + n1 - 2))
    return g, v

def bh_fdr(p_values):

    p = np.asarray(p_values, dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out

    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    ranked = pv[order] * m / np.arange(1, m + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    q = np.empty(m, dtype=float)
    q[order] = np.clip(ranked, 0, 1)
    out[ok] = q
    return out

def random_effects_dl(g, v):

    g = np.asarray(g, dtype=float)
    v = np.asarray(v, dtype=float)
    ok = np.isfinite(g) & np.isfinite(v) & (v > 0)
    g, v = g[ok], v[ok]
    k = len(g)
    if k < 2:
        return None

    w = 1.0 / v
    g_fixed = np.sum(w * g) / np.sum(w)
    Q = np.sum(w * (g - g_fixed)**2)
    df_q = k - 1
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (Q - df_q) / c) if c > 0 else 0.0

    w_re = 1.0 / (v + tau2)
    g_re = np.sum(w_re * g) / np.sum(w_re)
    se_re = np.sqrt(1.0 / np.sum(w_re))
    lo = g_re - 1.96 * se_re
    hi = g_re + 1.96 * se_re
    I2 = max(0.0, (Q - df_q) / Q) * 100 if Q > 0 else 0.0
    Q_p = chi2.sf(Q, df_q) if df_q > 0 else np.nan

    return {
        "g": g_re, "se": se_re, "v": se_re**2,
        "lo": lo, "hi": hi,
        "Q": Q, "Q_p": Q_p, "tau2": tau2, "I2": I2, "k": k,
        "p": 2 * norm.sf(abs(g_re / se_re)),
    }

def adjusted_smd_full(df_sub, y_col):

    required = [y_col, "_grp_", "age", "BMI", "gender"]
    dat = df_sub[required].copy()

    dat[y_col] = pd.to_numeric(dat[y_col], errors="coerce")
    dat["age"] = pd.to_numeric(dat["age"], errors="coerce")
    dat["BMI"] = pd.to_numeric(dat["BMI"], errors="coerce")
    dat["_sex_"] = gender_to_binary(dat["gender"])
    dat[[y_col, "age", "BMI"]] = dat[[y_col, "age", "BMI"]].replace([np.inf, -np.inf], np.nan)
    dat["_D_"] = (dat["_grp_"] == "Case").astype(float)

    if dat["age"].notna().sum() == 0:
        return None, "age unavailable"
    if dat["BMI"].notna().sum() == 0:
        return None, "BMI unavailable"
    if dat["_sex_"].notna().sum() == 0:
        return None, "sex unavailable or unrecognized"
    dat = dat.dropna(subset=[y_col, "_D_", "age", "BMI", "_sex_"]).copy()

    if dat.shape[0] <= 4:
        return None, "insufficient complete cases"
    if dat["_D_"].nunique() < 2:
        return None, "only one disease group after complete-case filtering"
    if (dat["_D_"] == 0).sum() < 2 or (dat["_D_"] == 1).sum() < 2:
        return None, "fewer than two observations in one group"
    if dat["age"].nunique() < 2:
        return None, "age has no variation"
    if dat["BMI"].nunique() < 2:
        return None, "BMI has no variation"
    sex_varies = dat["_sex_"].nunique() > 1
    design_columns = [
        np.ones(len(dat)),
        dat["_D_"].to_numpy(float),
        dat["age"].to_numpy(float),
        dat["BMI"].to_numpy(float),
    ]
    if sex_varies:
        design_columns.append(dat["_sex_"].to_numpy(float))
    X = np.column_stack(design_columns)
    y = dat[y_col].to_numpy(float)

    n, p = X.shape
    if np.linalg.matrix_rank(X) < p or n <= p:
        return None, "singular design matrix / insufficient residual df"

    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    df_resid = n - p
    s2 = float((resid @ resid) / df_resid)
    if not np.isfinite(s2) or s2 <= 0:
        return None, "non-positive residual variance"

    inv_xtx = np.linalg.inv(X.T @ X)
    cov_beta = s2 * inv_xtx
    bD = float(beta[1])
    se_bD = float(np.sqrt(cov_beta[1, 1]))
    s = float(np.sqrt(s2))

    if not np.isfinite(se_bD) or se_bD <= 0:
        return None, "invalid disease coefficient SE"

    d = bD / s
    J = 1 - 3 / (4 * df_resid - 1) if df_resid > 1 else 1.0
    g_adj = J * d

    v_adj = J**2 * ((se_bD / s)**2 + (d**2) / (2 * df_resid))
    t_stat = bD / se_bD
    p_value = 2 * t_dist.sf(abs(t_stat), df=df_resid)

    result = {
        "g": g_adj,
        "v": v_adj,
        "se": np.sqrt(v_adj),
        "p": p_value,
        "n_used": n,
        "n_ctrl": int((dat["_D_"] == 0).sum()),
        "n_case": int((dat["_D_"] == 1).sum()),
        "df_resid": df_resid,
        "n_female": int((dat["_sex_"] == 0).sum()),
        "n_male": int((dat["_sex_"] == 1).sum()),
        "sex_term_included": bool(sex_varies),
        "sex_handling": "estimated" if sex_varies else "constant in complete cases; redundant term omitted",
        "model_terms": "Case + age + BMI + sex" if sex_varies else "Case + age + BMI (sex constant)",
    }
    return result, None


In [ ]:
for path in (CSV_PATH, METADATA_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Cannot find input file: {path}")

abundance = pd.read_csv(CSV_PATH)
metadata = pd.read_excel(METADATA_PATH)
for data in (abundance, metadata):
    if "index" not in data.columns:
        raise ValueError("Both input files must contain an 'index' column.")
    data["index"] = data["index"].astype(str).str.strip()

df = abundance.merge(metadata, on="index", how="inner", validate="one_to_one")
if len(df) != len(abundance):
    raise ValueError(f"Index merge retained {len(df)} of {len(abundance)} abundance rows.")

required_cols = {"Cohort", "study_condition", "age", "BMI", "gender"}
missing_required = required_cols.difference(df.columns)
if missing_required:
    raise ValueError(f"Missing required metadata columns: {sorted(missing_required)}")

sex_encoded = gender_to_binary(df["gender"])
unknown_sex = df["gender"].notna() & sex_encoded.isna()
if unknown_sex.any():
    print("Unrecognized gender labels treated as missing:", df.loc[unknown_sex, "gender"].value_counts().to_dict())

features = [f for f in FEATURE_ORDER if FEATURE_SOURCE.get(f, f) in df.columns]
if not features:
    raise ValueError(f"None of the requested metabolites were found: {FEATURE_ORDER}")
missing_features = [f for f in FEATURE_ORDER if FEATURE_SOURCE.get(f, f) not in df.columns]
if missing_features:
    raise ValueError(
        f"Required metabolite columns are missing: {missing_features}. "
        f"Available columns are: {list(abundance.columns)}"
    )
for feat in features:
    source_feat = FEATURE_SOURCE.get(feat, feat)
    df[feat] = pd.to_numeric(df[source_feat], errors="coerce")
if FEATURE_SOURCE:
    print("Feature aliases applied:", {k: v for k, v in FEATURE_SOURCE.items() if k in features})

disease_data = {}
disease_summary_rows = []
for disease in DISEASE_ORDER:
    candidate = disease_subset(df, disease)
    counts = candidate.groupby(["_study_", "_grp_"]).size().unstack(fill_value=0)
    for group in ("Control", "Case"):
        if group not in counts.columns:
            counts[group] = 0
    eligible = counts.index[(counts["Control"] >= 2) & (counts["Case"] >= 2)]
    disease_summary_rows.append({
        "disease": disease,
        "eligible_cohorts": len(eligible),
        "cohorts": ", ".join(eligible.tolist()),
        "control_n": int(candidate.loc[candidate["_grp_"] == "Control"].shape[0]),
        "case_n": int(candidate.loc[candidate["_grp_"] == "Case"].shape[0]),
    })
    if len(eligible) < MIN_ELIGIBLE_COHORTS:
        print(f"Excluded {disease}: only {len(eligible)} eligible cohort(s).")
        continue
    disease_data[disease] = candidate[candidate["_study_"].isin(eligible)].copy()

disease_summary = pd.DataFrame(disease_summary_rows)
DISEASES = [d for d in DISEASE_ORDER if d in disease_data]
if len(DISEASES) != len(DISEASE_ORDER):
    raise ValueError(f"Expected four multi-cohort diseases, found: {DISEASES}")

print(f"Merged {len(df)} abundance rows with metadata.")
print("Metabolites plotted:", features)
display(disease_summary)

qc_rows = []
for disease, disease_df in disease_data.items():
    for study, sub in disease_df.groupby("_study_", sort=False):
        qc_rows.append({
            "disease": disease,
            "study": study,
            "N": len(sub),
            "age_nonmissing": pd.to_numeric(sub["age"], errors="coerce").notna().sum(),
            "BMI_nonmissing": pd.to_numeric(sub["BMI"], errors="coerce").notna().sum(),
            "sex_nonmissing": gender_to_binary(sub["gender"]).notna().sum(),
            "female_n": gender_to_binary(sub["gender"]).eq(0).sum(),
            "male_n": gender_to_binary(sub["gender"]).eq(1).sum(),
        })

covariate_qc = pd.DataFrame(qc_rows)
covariate_qc["eligible_in_principle"] = (
    (covariate_qc["age_nonmissing"] > 0) &
    (covariate_qc["BMI_nonmissing"] > 0) &
    (covariate_qc["sex_nonmissing"] > 0)
)

covariate_qc


In [ ]:
crude_by_disease = {}
for disease, disease_df in disease_data.items():
    crude_rows = []
    for feat in features:
        for study, sub in disease_df.groupby("_study_", sort=False):
            ctrl = sub.loc[sub["_grp_"] == "Control", feat].dropna().to_numpy(float)
            case = sub.loc[sub["_grp_"] == "Case", feat].dropna().to_numpy(float)
            g, v = hedges_g(ctrl, case)
            if not (np.isfinite(g) and np.isfinite(v) and v > 0):
                continue
            se = np.sqrt(v)
            crude_rows.append({
                "disease": disease,
                "feature": feat,
                "study": study,
                "g": g,
                "v": v,
                "se": se,
                "p": 2 * norm.sf(abs(g / se)),
                "n_ctrl": len(ctrl),
                "n_case": len(case),
            })
    crude = pd.DataFrame(crude_rows)
    if crude.empty:
        raise ValueError(f"No crude study-level effects could be calculated for {disease}.")
                                                                         
    crude["qval"] = np.nan
    for _, idx in crude.groupby("study", sort=False).groups.items():
        crude.loc[idx, "qval"] = bh_fdr(crude.loc[idx, "p"].to_numpy())
    crude_by_disease[disease] = crude

crude = pd.concat(crude_by_disease.values(), ignore_index=True)
display(crude.head())


In [ ]:
adjusted_by_disease = {}
adjusted_exclusions_by_disease = {}
for disease, disease_df in disease_data.items():
    adj_rows = []
    exclude_rows = []
    for feat in features:
        for study, sub in disease_df.groupby("_study_", sort=False):
            result, reason = adjusted_smd_full(sub, feat)
            if result is None:
                exclude_rows.append({"disease": disease, "feature": feat, "study": study, "reason": reason})
                continue
            adj_rows.append({"disease": disease, "feature": feat, "study": study, **result})
    adjusted = pd.DataFrame(adj_rows)
    adjusted_exclusions = pd.DataFrame(exclude_rows)
    if not adjusted.empty:
        adjusted["qval"] = np.nan
        for _, idx in adjusted.groupby("study", sort=False).groups.items():
            adjusted.loc[idx, "qval"] = bh_fdr(adjusted.loc[idx, "p"].to_numpy())
    adjusted_by_disease[disease] = adjusted
    adjusted_exclusions_by_disease[disease] = adjusted_exclusions

adjusted_parts = [x for x in adjusted_by_disease.values() if not x.empty]
adjusted_exclusion_parts = [x for x in adjusted_exclusions_by_disease.values() if not x.empty]
adjusted = pd.concat(adjusted_parts, ignore_index=True) if adjusted_parts else pd.DataFrame()
adjusted_exclusions = pd.concat(adjusted_exclusion_parts, ignore_index=True) if adjusted_exclusion_parts else pd.DataFrame()
print("Adjusted model exclusions:")
display(adjusted_exclusions if not adjusted_exclusions.empty else pd.DataFrame(columns=["disease", "feature", "study", "reason"]))


In [ ]:
def pool_by_feature(effect_df, feature_order=features):
    rows = []
    if effect_df is None or effect_df.empty:
        return pd.DataFrame()

    for feat in feature_order:
        sub = effect_df[effect_df["feature"] == feat]
        meta = random_effects_dl(sub["g"].to_numpy(), sub["v"].to_numpy())
        if meta is None:
            continue
        rows.append({"feature": feat, **meta})

    pooled = pd.DataFrame(rows)
    if not pooled.empty:
        pooled["qval"] = bh_fdr(pooled["p"].to_numpy())
    return pooled

pooled_crude_by_disease = {
    disease: pool_by_feature(crude_by_disease[disease])
    for disease in DISEASES
}
pooled_adjusted_by_disease = {
    disease: pool_by_feature(adjusted_by_disease[disease])
    for disease in DISEASES
}
pooled_crude = pd.concat([x.assign(disease=disease) for disease, x in pooled_crude_by_disease.items()], ignore_index=True)
pooled_adjusted_parts = [x.assign(disease=disease) for disease, x in pooled_adjusted_by_disease.items() if not x.empty]
pooled_adjusted = pd.concat(pooled_adjusted_parts, ignore_index=True) if pooled_adjusted_parts else pd.DataFrame()

print("Crude pooled results")
display_cols = ["feature", "g", "lo", "hi", "p", "qval", "Q", "Q_p", "tau2", "I2", "k"]
display(pooled_crude[["disease"] + display_cols] if not pooled_crude.empty else pooled_crude)

print("Adjusted pooled results")
display(pooled_adjusted[["disease"] + display_cols] if not pooled_adjusted.empty else pooled_adjusted)


In [ ]:
def plot_reference_style(
    crude_df,
    pooled_crude_df,
    pooled_adjusted_df,
    feature_order,
    n_control,
    n_case,
    disease="IBD",
    q_cut=0.10,
    xlim=(-1.25, 1.50),
):
                            
    blue = "#4A78B8"                                       
    gold = "#D9A31A"                                         
    red = "#9E1B1B"                                          
    gray = "#B8B8B8"                                           
    light_gray = "#E2E2E2"                                 
    dark = "#303030"
    grid = "#D9D9D9"
    arrow_blue = "#0072BC"

    features_present = [f for f in feature_order if f in pooled_crude_df["feature"].tolist()]
    if not features_present:
        raise ValueError("No pooled crude features available for plotting.")
                                                                              
    crude_g = pooled_crude_df.set_index("feature")["g"]
    features_present.sort(key=lambda f: (crude_g.loc[f], f), reverse=True)

    y_map = {feat: i for i, feat in enumerate(features_present)}
    n_rows = len(features_present)

    fig_h = max(4.8, 0.72 * n_rows + 2.2)
    fig, ax = plt.subplots(figsize=(4.7, fig_h))

    ax.set_xlim(*xlim)
    ax.set_ylim(-0.55, n_rows - 0.45)
    ax.axvline(0, color=dark, lw=1.0, zorder=0)
    ax.xaxis.set_major_locator(MultipleLocator(0.5))
    ax.xaxis.set_minor_locator(MultipleLocator(0.25))
    ax.grid(True, axis="x", which="major", color=grid, lw=0.65, alpha=0.75)
    ax.grid(True, axis="x", which="minor", color=grid, lw=0.45, alpha=0.50)
    ax.grid(True, axis="y", which="major", color=grid, lw=0.65, alpha=0.75)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.25)
        spine.set_color(dark)

    study_order = sorted(
        crude_df["study"].dropna().unique().tolist(),
        key=lambda s: COHORT_DISPLAY.get(s, s).casefold(),
    )
    if len(study_order) > len(STUDY_MARKER_SET):
        raise ValueError(f"Add more simple markers for {len(study_order)} cohorts.")
    study_markers = {study: STUDY_MARKER_SET[i] for i, study in enumerate(study_order)}

    for feat in features_present:
        y0 = y_map[feat]
        sub = crude_df[crude_df["feature"] == feat].copy()
        sub["_ord_"] = sub["study"].map({s: i for i, s in enumerate(study_order)})
        sub = sub.sort_values("_ord_")

        if len(sub) == 1:
            offsets = [0.0]
        else:
            offsets = np.linspace(-0.16, 0.16, len(sub))

        for off, (_, r) in zip(offsets, sub.iterrows()):
            lo = r["g"] - 1.96 * r["se"]
            hi = r["g"] + 1.96 * r["se"]
            sig = bool(r.get("qval", 1.0) < q_cut)
            c = blue if sig else gray
            marker = study_markers.get(r["study"], "o")
            yy = y0 + off
                                                                                                     
            ax.plot([lo, hi], [yy, yy], color=c, lw=1.25, alpha=0.75, zorder=6)
            ax.scatter(
                r["g"], yy,
                marker=marker, s=70,
                facecolor=c, edgecolor="white", linewidth=0.7,
                alpha=0.65, zorder=7,
            )

    crude_lookup = pooled_crude_df.set_index("feature")
    for feat in features_present:
        r = crude_lookup.loc[feat]
        y = y_map[feat]
        sig = bool(r.get("qval", 1.0) < q_cut)
        face = gold if sig else light_gray
        diamond = np.array([
            [r["lo"], y],
            [r["g"], y + 0.115],
            [r["hi"], y],
            [r["g"], y - 0.115],
        ])
        ax.add_patch(Polygon(
            diamond, closed=True,
            facecolor=face, edgecolor=dark, lw=1.0, zorder=3
        ))

    if pooled_adjusted_df is not None and not pooled_adjusted_df.empty:
        adj_lookup = pooled_adjusted_df.set_index("feature")
        for feat in features_present:
            if feat not in adj_lookup.index:
                continue
            r = adj_lookup.loc[feat]
            y = y_map[feat]
            sig = bool(r.get("qval", 1.0) < q_cut)
            face = red if sig else "white"
            edge = dark
            ci_color = red if sig else gray
            ax.plot([r["lo"], r["hi"]], [y, y], color=ci_color, lw=1.15, zorder=4)
            ax.scatter(
                r["g"], y,
                marker="*", s=170,
                facecolor=face, edgecolor=edge, linewidth=0.8,
                zorder=10,
            )

    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(features_present, fontsize=11, fontweight="bold")
    ax.invert_yaxis()

    x_right = xlim[1]
    x_span = xlim[1] - xlim[0]
    for feat in features_present:
        r = crude_lookup.loc[feat]
        ax.text(
            x_right + 0.055 * x_span, y_map[feat],
            f"{r['I2']:.0f}%",
            va="center", ha="left", fontsize=10, color=dark,
            clip_on=False,
        )

    ax.set_title(
        f"{disease}   H/C = {n_control}/{n_case}",
        fontsize=14, fontweight="bold", pad=7,
    )

    ax.tick_params(axis="x", labelsize=10)
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("")

    y_arrow = -0.105
    ax.annotate(
        "", xy=(0.06, y_arrow), xytext=(0.42, y_arrow),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->", lw=1.4, color=arrow_blue),
        annotation_clip=False,
    )
    ax.annotate(
        "", xy=(0.94, y_arrow), xytext=(0.58, y_arrow),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->", lw=1.4, color=arrow_blue),
        annotation_clip=False,
    )
    ax.text(0.10, -0.17, "Control", transform=ax.transAxes,
            color=arrow_blue, fontsize=11, fontweight="bold", ha="left")
    ax.text(0.50, -0.17, "SMD", transform=ax.transAxes,
            color=arrow_blue, fontsize=11, fontweight="bold", ha="center")
    ax.text(0.84, -0.17, disease, transform=ax.transAxes,
            color=arrow_blue, fontsize=11, fontweight="bold", ha="center")

    study_handles = []
    for study in study_order:
        study_handles.append(Line2D(
            [0], [0], marker=study_markers[study], linestyle="None",
            markerfacecolor="white", markeredgecolor=dark,
            markersize=7, label=COHORT_DISPLAY.get(study, study),
        ))
    leg1 = ax.legend(
        handles=study_handles,
        loc="lower right", bbox_to_anchor=(0.99, 0.02),
        frameon=False, fontsize=9,
        handlelength=0.9, handletextpad=0.35, borderaxespad=0.0,
    )
    ax.add_artist(leg1)

    crude_model = Line2D([0], [0], marker="D", linestyle="None",
                         markerfacecolor=dark, markeredgecolor=dark, markersize=7)
    adjusted_model = Line2D([0], [0], marker="*", linestyle="None",
                            markerfacecolor=dark, markeredgecolor=dark, markersize=10)

    ns_tuple = (
        Line2D([0], [0], marker="^", linestyle="None", markerfacecolor=gray,
               markeredgecolor="white", markersize=7),
        Line2D([0], [0], marker="D", linestyle="None", markerfacecolor=light_gray,
               markeredgecolor=dark, markersize=7),
        Line2D([0], [0], marker="*", linestyle="None", markerfacecolor="white",
               markeredgecolor=dark, markersize=9),
    )
    sig_tuple = (
        Line2D([0], [0], marker="^", linestyle="None", markerfacecolor=blue,
               markeredgecolor="white", markersize=7),
        Line2D([0], [0], marker="D", linestyle="None", markerfacecolor=gold,
               markeredgecolor=dark, markersize=7),
        Line2D([0], [0], marker="*", linestyle="None", markerfacecolor=red,
               markeredgecolor=dark, markersize=9),
    )

    fig.legend(
        [crude_model, adjusted_model, ns_tuple, sig_tuple],
        ["Crude Model", "Adjusted Model", r"$q \geq 0.1$", r"$q < 0.1$"],
        handler_map={tuple: HandlerTuple(ndivide=None)},
        loc="lower center", bbox_to_anchor=(0.5, 0.005),
        ncol=2, frameon=True, fancybox=False, edgecolor=dark,
        fontsize=9, columnspacing=1.1, handletextpad=0.55,
    )

    fig.subplots_adjust(left=0.18, right=0.82, top=0.90, bottom=0.25)
    return fig, ax

output_dir = CSV_PATH.parent / "age_sex_BMI_adjusted"
output_dir.mkdir(parents=True, exist_ok=True)
figures = {}
figure_paths = {}
for disease in DISEASES:
    disease_df = disease_data[disease]
    n_control = int((disease_df["_grp_"] == "Control").sum())
    n_case = int((disease_df["_grp_"] == "Case").sum())
    fig, ax = plot_reference_style(
        crude_df=crude_by_disease[disease],
        pooled_crude_df=pooled_crude_by_disease[disease],
        pooled_adjusted_df=pooled_adjusted_by_disease[disease],
        feature_order=FEATURE_ORDER,
        n_control=n_control,
        n_case=n_case,
        disease=disease,
        q_cut=Q_CUT,
        xlim=XLIM,
    )
    stem = f"{OUTPUT_STEM}_{disease}"
    svg_path = output_dir / f"{stem}.svg"
    png_path = output_dir / f"{stem}.png"
    fig.savefig(svg_path, format="svg", bbox_inches="tight", transparent=True)
    fig.savefig(png_path, format="png", dpi=600, bbox_inches="tight", facecolor="white")
    figures[disease] = (fig, ax)
    figure_paths[disease] = {"svg": svg_path, "png": png_path}
    plt.show()
    print(f"{disease}: saved {svg_path.name} and {png_path.name}")


In [ ]:
for disease in DISEASES:
    stem = f"{OUTPUT_STEM}_{disease}"
    crude_by_disease[disease].to_csv(output_dir / f"{stem}_study_crude.csv", index=False)
    pooled_crude_by_disease[disease].to_csv(output_dir / f"{stem}_pooled_crude.csv", index=False)
    if not adjusted_by_disease[disease].empty:
        adjusted_by_disease[disease].to_csv(output_dir / f"{stem}_study_adjusted.csv", index=False)
    if not pooled_adjusted_by_disease[disease].empty:
        pooled_adjusted_by_disease[disease].to_csv(output_dir / f"{stem}_pooled_adjusted.csv", index=False)
    if not adjusted_exclusions_by_disease[disease].empty:
        adjusted_exclusions_by_disease[disease].to_csv(output_dir / f"{stem}_adjusted_exclusions.csv", index=False)

crude.to_csv(output_dir / f"{OUTPUT_STEM}_all_study_crude.csv", index=False)
pooled_crude.to_csv(output_dir / f"{OUTPUT_STEM}_all_pooled_crude.csv", index=False)
if not adjusted.empty:
    adjusted.to_csv(output_dir / f"{OUTPUT_STEM}_all_study_adjusted.csv", index=False)
if not pooled_adjusted.empty:
    pooled_adjusted.to_csv(output_dir / f"{OUTPUT_STEM}_all_pooled_adjusted.csv", index=False)
if not adjusted_exclusions.empty:
    adjusted_exclusions.to_csv(output_dir / f"{OUTPUT_STEM}_all_adjusted_exclusions.csv", index=False)

covariate_qc.to_csv(output_dir / f"{OUTPUT_STEM}_covariate_availability.csv", index=False)
print(f"Exported four disease panels and result tables to: {output_dir}")
